# StatsBomb–Transfermarkt Join Prototype

Phase 1 — Data Inspection, part 3. This notebook proves the player join works end to end on one test season: La Liga 2015/16 on both sides (380 StatsBomb matches vs Transfermarkt's Spanish-league appearances from July 2015 to June 2016).

Known hurdle: StatsBomb uses full legal names (Luis Alberto Suarez Diaz) while Transfermarkt uses everyday names (Neymar), so exact matching will underperform — the method under test matches on shared name-words plus club agreement.

## Cell 1 — Setup and scope

Folder paths, the season under test on both sides, and existence checks. Accent-stripping uses Python's built-in tools; nothing new to install.

In [1]:
import json, os, re, unicodedata
import pandas as pd
import numpy as np

TM_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'transfermarkt'))
SB_DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'statsbomb', 'data'))
SB_COMP, SB_SEASON = 11, 27
SEASON_START, SEASON_END = '2015-07-01', '2016-06-30'

print('tm exists:', os.path.isdir(TM_DIR))
print('sb exists:', os.path.isdir(SB_DATA_DIR))


tm exists: True
sb exists: True


## Cell 2 — StatsBomb side: players per team

Reads all 380 lineup files and reduces to one row per player-team pair. Match numbers come from the matches file; names come from the lineups.

In [2]:
ms = json.load(open(os.path.join(SB_DATA_DIR, 'matches', str(SB_COMP), f'{SB_SEASON}.json'), encoding='utf-8'))
print('matches:', len(ms))
sb_rows = []
for m in ms:
    fp = os.path.join(SB_DATA_DIR, 'lineups', f"{m['match_id']}.json")
    for t in json.load(open(fp, encoding='utf-8')):
        for p in t['lineup']:
            sb_rows.append((p['player_name'], t['team_name']))
df_sb = pd.DataFrame(sb_rows, columns=['sb_player', 'sb_team']).drop_duplicates()
print('distinct SB player-team pairs:', len(df_sb), '| players:', df_sb['sb_player'].nunique())
df_sb.head()


matches: 380
distinct SB player-team pairs: 612 | players: 601


,sb_player,sb_team
0,Danilo Luiz da Silva,Real Madrid
1,Jesé Rodríguez Ruiz,Real Madrid
2,Francisco Román Alarcón Suárez,Real Madrid
3,Cristiano Ronaldo dos Santos Aveiro,Real Madrid
4,Mateo Kovačić,Real Madrid


## Cell 3 — Transfermarkt side: players per club, correct for that season

Filters appearances to Spanish-league games in the season window and translates club IDs to names. Season-windowed appearances are used instead of the current-club snapshot (which shows today's squads), so club links are right for 2015/16.

In [3]:
app = pd.read_csv(os.path.join(TM_DIR, 'appearances.csv'),
                  usecols=['player_id', 'player_name', 'player_club_id', 'date', 'competition_id'],
                  parse_dates=['date'])
es = app[(app['competition_id'] == 'ES1') & (app['date'] >= SEASON_START) & (app['date'] < '2016-07-01')]
clubs = pd.read_csv(os.path.join(TM_DIR, 'clubs.csv'), usecols=['club_id', 'name'])
df_tm = (es[['player_id', 'player_name', 'player_club_id']].drop_duplicates()
           .merge(clubs, left_on='player_club_id', right_on='club_id')
           .rename(columns={'name': 'tm_club'})[['player_id', 'player_name', 'tm_club']])
print('distinct TM player-club pairs:', len(df_tm), '| players:', df_tm['player_name'].nunique())
df_tm.head()


distinct TM player-club pairs: 545 | players: 536


,player_id,player_name,tm_club
0,105521,Ciro Immobile,Sevilla FC
1,12249,Éver Banega,Sevilla FC
2,126625,Juankar,Málaga CF
3,13038,Arthur Boka,Málaga CF
4,142021,Sergi Darder,Málaga CF


## Cell 4 — Name cleaning

Lowercases, strips accents and punctuation on both name columns. All matching from here works on cleaned keys, never raw names.

In [4]:
def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'[^a-z ]', '', s.lower()).strip()

df_sb['sb_key'] = df_sb['sb_player'].apply(norm)
df_tm['tm_key'] = df_tm['player_name'].apply(norm)
print(df_sb[['sb_player', 'sb_key']].head(5).to_string())
print()
print(df_tm[['player_name', 'tm_key']].head(5).to_string())


                             sb_player                               sb_key
0                 Danilo Luiz da Silva                 danilo luiz da silva
1                  Jesé Rodríguez Ruiz                  jese rodriguez ruiz
2       Francisco Román Alarcón Suárez       francisco roman alarcon suarez
3  Cristiano Ronaldo dos Santos Aveiro  cristiano ronaldo dos santos aveiro
4                        Mateo Kovačić                        mateo kovacic

     player_name         tm_key
0  Ciro Immobile  ciro immobile
1    Éver Banega    ever banega
2        Juankar        juankar
3    Arthur Boka    arthur boka
4   Sergi Darder   sergi darder


## Cell 5 — Exact-match baseline (expected to miss a lot)

Joins on identical cleaned names. Stars will miss (long legal name vs short common name); the miss list motivates Cell 6.

In [5]:
exact = df_sb.merge(df_tm, left_on='sb_key', right_on='tm_key', how='left', indicator=True)
print('SB pairs with exact normalized-name match:', (exact['_merge'] == 'both').sum(), '/', len(exact))
print()
print('sample misses:', exact.loc[exact['_merge'] == 'left_only', 'sb_player'].head(10).tolist())


SB pairs with exact normalized-name match: 88 / 614

sample misses: ['Danilo Luiz da Silva', 'Jesé Rodríguez Ruiz', 'Francisco Román Alarcón Suárez', 'Cristiano Ronaldo dos Santos Aveiro', 'Carlos Henrique Casimiro', 'Marcelo Vieira da Silva Júnior', 'Keylor Navas Gamboa', 'James David Rodríguez Rubio', 'Daniel Carvajal Ramos', 'Gareth Frank Bale']


## Cell 6 — Word-based matching plus club check

A Transfermarkt name matches when every one of its words appears in the StatsBomb name (lionel + messi are both inside lionel, andres, messi, cuccittini). One-word common names can hit several players — counted as ambiguous, not hidden. Club-name agreement is the acceptance test: matching names at different clubs stay unmatched, while same-club spelling variants (Barcelona vs FC Barcelona) pass.

In [6]:
sb_tok = df_sb.assign(tok=df_sb['sb_key'].str.split())
tm_tok = df_tm.assign(tok=df_tm['tm_key'].str.split())
pairs = sb_tok.merge(tm_tok, how='cross', suffixes=('_sb', '_tm'))
pairs = pairs[pairs.apply(lambda r: set(r['tok_tm']).issubset(set(r['tok_sb'])), axis=1)]
print('token-subset pairs:', len(pairs))
n_sb = df_sb['sb_player'].nunique()
matched = pairs['sb_player'].nunique()
multi = (pairs.groupby('sb_player')['player_id'].nunique() > 1).sum()
print(f'SB players: {n_sb} total | {matched} matched | {multi} ambiguous (2+ TM candidates)')
print()
pairs['club_ok'] = pairs.apply(
    lambda r: norm(r['sb_team']) in norm(r['tm_club']) or norm(r['tm_club']) in norm(r['sb_team']), axis=1)
print('pairs with club agreement:', pairs['club_ok'].sum(), '/', len(pairs))
print()
print('club mismatches to map manually:')
print(pairs.loc[~pairs['club_ok'], ['sb_player', 'sb_team', 'player_name', 'tm_club']]
      .drop_duplicates().head(10).to_string())


token-subset pairs: 473
SB players: 601 total | 439 matched | 16 ambiguous (2+ TM candidates)

pairs with club agreement: 356 / 473

club mismatches to map manually:
                              sb_player                 sb_team         player_name                 tm_club
10680       Jonathan Rodríguez Menéndez          Sporting Gijón  Jonathan Rodríguez  Deportivo de La Coruña
14451               Sergio Álvarez Díaz          Sporting Gijón      Sergio Álvarez           Celta de Vigo
39375          Fernando Llorente Torres                 Sevilla     Fernando Torres      Atlético de Madrid
45025  Sebastián Carlos Cristóforo Pepe                 Sevilla                Pepe             Real Madrid
56413       Juan Carlos Valerón Santana              Las Palmas         Juan Carlos          Rayo Vallecano
58967                 Celso Borges Mora  RC Deportivo La Coruña        Celso Borges  Deportivo de La Coruña
59492                       Fayçal Fajr  RC Deportivo La Coruña         Fayçal

## Cell 7 — Attaching the price

For club-agreed matches, takes each player's latest price on or before season end — the prototype player-season-price row. Ambiguous players keep their first candidate here; reviewing them is booked as follow-up work, not decided silently.

In [7]:
best = pairs[pairs['club_ok']].sort_values('player_id').drop_duplicates('sb_player')
val = pd.read_csv(os.path.join(TM_DIR, 'player_valuations.csv'), parse_dates=['date'])
val = val[val['date'] <= SEASON_END].sort_values('date').drop_duplicates('player_id', keep='last')
out = best.merge(val[['player_id', 'date', 'market_value_in_eur']], on='player_id', how='left')
print('matched with club agreement:', len(best), '| with valuation:', out['market_value_in_eur'].notna().sum())
print()
print(out[['sb_player', 'sb_team', 'player_name', 'tm_club', 'market_value_in_eur']].head(15).to_string())


matched with club agreement: 353 | with valuation: 341

                        sb_player         sb_team           player_name              tm_club  market_value_in_eur
0   Roque Luis Santa Cruz Cantero          Málaga      Roque Santa Cruz            Málaga CF            1000000.0
1                Heiko Westermann      Real Betis      Heiko Westermann  Real Betis Balompié            1500000.0
2            Rafael van der Vaart      Real Betis  Rafael van der Vaart  Real Betis Balompié            1000000.0
3                  Jérémy Mathieu       Barcelona        Jérémy Mathieu         FC Barcelona           10000000.0
4                  Daniele Bonera      Villarreal        Daniele Bonera        Villarreal CF             500000.0
5                    Henok Goitom          Getafe          Henok Goitom            Getafe CF            1200000.0
6     Juan Carlos Valerón Santana      Las Palmas   Juan Carlos Valerón        UD Las Palmas             100000.0
7            David Navarro Pedró

## Next steps

1. Build the manual club-name map from Cell 6's mismatches.
2. Resolve ambiguous players with birth dates and positions.
3. Fuzzy-match the leftover names.
4. Scale the proven path to all shared league-seasons, then design Postgres.